In [ ]:
from pathlib import Path

import numpy as np
from uncertainties import ufloat

from betata import plt
from betata.qubit_measurements.qubit import load_qubits, save_qubit
from betata.qubit_measurements.traces import load_t1_traces, load_t2e_traces

CWD = Path.cwd()

# t1 and t2e traces taken longer TIME_DIFF_THRESHOLD seconds apart are not paired
TIME_DIFF_THRESHOLD = 300  # seconds

# reject t_phi if percentage error exceeds T_PHI_ERR_THRESHOLD
T_PHI_ERR_THRESHOLD = 0.5

In [ ]:
all_qubits = load_qubits()

Helper functions

In [ ]:
def pair_t1_t2e_traces(t1_traces, t2e_traces):
    """ """
    t1_t2e_trace_pairs = {} # key: timestamp, value: (t1_trace_id, t2e_trace_id)
    for t1_tr in t1_traces:
        time_dx = [np.abs(t1_tr.timestamp - t2e_tr.timestamp) for t2e_tr in t2e_traces]
        min_time_diff, arg_min_time_diff = np.min(time_dx), np.argmin(time_dx)
        if min_time_diff.total_seconds() <= TIME_DIFF_THRESHOLD:
            t2e_tr = t2e_traces[arg_min_time_diff]
            t_phi_timestamp = t1_tr.timestamp + (t2e_tr.timestamp - t1_tr.timestamp) / 2
            t1_t2e_trace_pairs[t_phi_timestamp] = (t1_tr.id, t2e_tr.id)
    return t1_t2e_trace_pairs

In [ ]:
def t_phi_fn(t2e, t1):
    """ """
    gamma_phi = ((1 / t2e) - (1 / (2 * t1)))
    return 1 / gamma_phi if gamma_phi else np.inf

In [ ]:
def save_t_phi_data(qubit, t1_t2e_trace_pairs, plot=False):
    """ """

    t_phi = []
    t_phi_err = []
    t_phi_trace_id = []
    t_phi_timestamp = []  # associate with the midpoint of the t1 and t2e timestamp

    for timestamp, (t1_tr_id, t2e_tr_id) in t1_t2e_trace_pairs.items():
        t1_idx = np.argwhere(qubit.t1_trace_id == t1_tr_id)[0][0]
        t2e_idx = np.argwhere(qubit.t2e_trace_id == t2e_tr_id)[0][0]
        t1 = ufloat(qubit.t1[t1_idx], qubit.t1_err[t1_idx])
        t2e = ufloat(qubit.t2e[t2e_idx], qubit.t2e_err[t2e_idx])
        tphi = t_phi_fn(t2e, t1)

        if tphi.n <= 0 or abs(tphi.s / tphi.n) > T_PHI_ERR_THRESHOLD:
            continue

        t_phi.append(tphi.n)
        t_phi_err.append(tphi.s)
        t_phi_trace_id.append([t1_tr_id, t2e_tr_id])

        t_phi_timestamp.append(timestamp)

    qubit.t_phi = np.array(t_phi)
    qubit.t_phi_err = np.array(t_phi_err)
    qubit.t_phi_trace_id = np.array(t_phi_trace_id)
    qubit.t_phi_timestamp = np.array(
        [np.abs((ts - t_phi_timestamp[0]).total_seconds()) for ts in t_phi_timestamp]
    )
    qubit.t_phi_avg = np.mean(t_phi)
    qubit.t_phi_avg_err = np.std(t_phi)

    if t_phi:
        save_qubit(qubit)

    if t_phi and plot:
        plot_t_phi_vs_time(qubit.t_phi, qubit.t_phi_err, t_phi_timestamp, qubit.name)

In [ ]:
def plot_t_phi_vs_time(tphi, tphi_err, timestamps, qubit_name: str):
    """ """

    t_phi_timestamps_hr = np.array(
        [(timestamp - timestamps[0]).total_seconds() / 3600 for timestamp in timestamps]
    )

    tphi_us, tphi_err_us = tphi * 1e6, tphi_err * 1e6

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.errorbar(
        t_phi_timestamps_hr,
        tphi_us,
        yerr=tphi_err_us,
        ls="",
        color="k",
        marker="o",
        alpha=0.85,
        zorder=-1,
    )

    tphi_avg, tphi_avg_err = np.mean(tphi), np.std(tphi)
    tphi_avg_us, tphi_avg_err_us = tphi_avg * 1e6, tphi_avg_err * 1e6

    ax.axhline(tphi_avg_us, ls="--", color="r")
    ax.axhline(tphi_avg_us - tphi_avg_err_us / 2, ls="--", color="r", alpha=0.5)
    ax.axhline(tphi_avg_us + tphi_avg_err_us / 2, ls="--", color="r", alpha=0.5)

    avg_tphi_str = r"$\overline{T}_\phi = $"
    avg_tphi_str += f"{tphi_avg_us:.1f} ± {tphi_avg_err_us:.1f} μs"
    ax.set_title(f"{qubit_name}: {avg_tphi_str}")

    ax.set_xlabel("Time (hour)")
    ax.set_ylabel(r"$T_\phi$ (μs)")

    fig.tight_layout()

    return fig

Calculate T_phi and save data to qubit file

In [ ]:
for qubit in all_qubits:

    data_folder = CWD / f"data/qubit_measurements/{qubit.name}"

    # load traces
    t1_traces = load_t1_traces(data_folder / f"T1_{qubit.name}")
    t2e_traces = load_t2e_traces(data_folder/ f"T2E_{qubit.name}")

    # filter traces
    t1_traces = [tr for tr in t1_traces if tr.id in qubit.t1_trace_id]
    t2e_traces = [tr for tr in t2e_traces if tr.id in qubit.t2e_trace_id]

    # pair traces
    trace_pairs = pair_t1_t2e_traces(t1_traces, t2e_traces)

    save_t_phi_data(qubit, trace_pairs, plot=True)

Plot T1 vs TPHI

In [ ]:
def plot_t_phi_vs_t1(qubit):
    """ """
    t_phi_us = qubit.t_phi * 1e6

    if not len(t_phi_us):
        print(f"{qubit.name} NO DATA AVAILABLE")
        return

    t_phi_err_us = qubit.t_phi_err * 1e6
    t_phi_trace_id_t1 = qubit.t_phi_trace_id[:, 0]

    t1 = []
    t1_err = []
    t1_trace_id_list = list(qubit.t1_trace_id)
    for tr_id in t_phi_trace_id_t1:
        t1_tr_idx = t1_trace_id_list.index(tr_id)
        t1.append(qubit.t1[t1_tr_idx])
        t1_err.append(qubit.t1_err[t1_tr_idx])
    t1_us = np.array(t1) * 1e6
    t1_err_us = np.array(t1_err) * 1e6

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.errorbar(
        t1_us,
        t_phi_us,
        xerr=t1_err_us,
        yerr=t_phi_err_us,
        ls="",
        color="k",
        marker="o",
        alpha=0.85,
        zorder=-1,
    )

    ax.set_title(f"{qubit.name}")

    ax.set_xlabel(r"$T_\mathrm{1}$ (μs)")
    ax.set_ylabel(r"$T_\phi$ (μs)")

    #t_min, t_max = min(min(t1_us), min(t_phi_us)), max(max(t1_us), max(t_phi_us))
    #dummy_time = np.linspace(t_min, t_max, 1001)
    #ax.plot(dummy_time, dummy_time, ls="--", c="k")

    fig.tight_layout()

    return fig

In [ ]:
for qubit in all_qubits:
    fig = plot_t_phi_vs_t1(qubit)

In [ ]:
for qubit in all_qubits:
    tphi_avg_us = qubit.t_phi_avg * 1e6
    tphi_avg_err_us = qubit.t_phi_avg_err * 1e6
    avg_tphi_str = f"{tphi_avg_us:.1f} ± {tphi_avg_err_us:.1f} μs"
    print(f"{qubit.name} {avg_tphi_str}, N = {len(qubit.t_phi)}")

In [ ]:
mean_q_avg = np.mean([qubit.q_avg for qubit in all_qubits])
mean_q_avg_err = np.std([qubit.q_avg for qubit in all_qubits])
print(mean_q_avg, mean_q_avg_err)

In [ ]:
t1s = np.array([216 , 328, 304 , 123 , 159 , 338 , 256 , 175 , 185 , 191 ,  78 ,])

In [ ]:
np.std(t1s)

In [ ]:
for qubit in all_qubits:
    print(qubit.name)
    print(qubit.q_avg)